# `klygo.datasets.load`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/datasets/Document_Datasets_Load.ipynb)

Load flat YOLO images and bounding boxes into in-memory `Detections` outputs with coordinates mapped to pixels for inspection and cropping.

## Jump to an API

- [`datasets.load`](#datasetsload)


In [1]:
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release


## Setup workspace and sample data


In [2]:
from tempfile import TemporaryDirectory
from PIL import Image
from klygo import datasets, files, media

# Create isolated tutorial workspace
tutorial_workspace = TemporaryDirectory(prefix="klygo-datasets-tut-")
root = files.path(tutorial_workspace.name)

# Create a sample flat YOLO dataset
sample_dataset = files.join(root, "sample_dataset")
images_dir = files.join(sample_dataset, "images")
labels_dir = files.join(sample_dataset, "labels")
files.mkdir(images_dir)
files.mkdir(labels_dir)

for i in range(4):
    cls = i % 2
    media.save(files.join(images_dir, f"img_{i}.jpg"), Image.new("RGB", (64, 48), color=(40 * i, 80, 120)), verbose=False)
    files.save(files.join(labels_dir, f"img_{i}.txt"), [f"{cls} 0.5 0.5 0.3 0.3"], verbose=False)

files.save(
    files.join(sample_dataset, "data.yaml"),
    {"names": {0: "cat", 1: "dog"}, "nc": 2},
    verbose=False,
)
print("Sample dataset ready:", sample_dataset)


Sample dataset ready: C:\Users\Mine\AppData\Local\Temp\klygo-datasets-tut-r7666ku6\sample_dataset


### datasets.load

Demo and practical usage for `datasets.load`.


In [3]:
# Load the flat dataset into Detections
detections = datasets.load(sample_dataset, format="yolo", verbose=False)
print("Loaded frames count:", len(detections))
print("First frame object count:", len(detections[0].objects))
print("First frame box in pixels:", detections[0].boxes)
print("First image size:", detections[0].image_size)


Loaded frames count: 4
First frame object count: 1
First frame box in pixels: [(22.4, 16.799999999999997, 41.6, 31.200000000000003)]
First image size: (64, 48)


## Put it together


In [4]:
# Complete end-to-end workflow demonstrating datasets.load
print("Executing complete workflow for datasets.load...")
stats = datasets.info(sample_dataset, format="yolo")
print("Dataset classes:", stats["classes"])
print("Dataset images:", stats["total_images"])


Executing complete workflow for datasets.load...
Dataset classes: ['cat', 'dog']
Dataset images: 4


## Load named annotation sets

A single string returns `Detections`. Multiple label directories return the same
`outputs.DetectionGroup` as `outputs.group`. Two sequence entries use `pred`,
`truth` in input order; three or more use `set_1`, `set_2`, and so on. A mapping
preserves explicit names. Shared operations mutate the original collections.


In [ ]:
from klygo import outputs, metrics
files.copy(files.join(sample_dataset, "labels"), files.join(sample_dataset, "pseudo_labels"), overwrite=True)
sets = datasets.load(sample_dataset, labels_dir=["pseudo_labels", "labels"], verbose=False)
assert isinstance(sets, outputs.DetectionGroup)
assert list(sets.keys()) == ["pred", "truth"]
sets.astype(32).pack().unpack()
expected = metrics.evaluate(sets["pred"], sets["truth"])
try:
    import cupy as cp
    cuda_ready = cp.cuda.runtime.getDeviceCount() > 0
except (ImportError, RuntimeError):
    cuda_ready = False
if cuda_ready:
    sets.cuda(0)
    assert metrics.evaluate(sets["pred"], sets["truth"]) == expected
    sets.cpu()
named = datasets.load(sample_dataset, labels_dir={"model": "pseudo_labels", "gt": "labels"}, verbose=False)
assert list(named.keys()) == ["model", "gt"]
print(expected)


## Remember

`datasets.load()` expects a flat dataset (call `datasets.unpartition()` first if your data is partitioned). It decodes pixels into RAM so images and crops survive archive extraction cleanup.

See [Datasets overview](Document_Datasets_Overview.ipynb) ([Colab](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/datasets/Document_Datasets_Overview.ipynb)) for full module documentation.


In [5]:
tutorial_workspace.cleanup()


Done.
